# 🧭 Ontologies as Guardrails for AI Agents
### Student edition · Compass AI · Budapest workshop · 90 minutes · Dr. Frank Coyle ("Dr. C")

An LLM agent is fluent but has no built-in sense of *what it is allowed to do*.
An **ontology** gives the agent's world a formal shape — and that shape can be used to
**check, infer, and block** the agent's actions before they happen.

**Our running scenario:** a customer-support AI agent (`botX`) at a webshop that can approve refunds.
What could possibly go wrong? 😅

| Part | Topic | What it gives a guardrail | Time |
|---|---|---|---|
| A | Triples & graphs | a shared, explicit picture of the world | 10 min |
| B | RDFS domain & range | *types* the agent never stated | 15 min |
| C | OWL transitive & inverse | chains of authority, two-way links | 15 min |
| D | SPARQL SELECT & ASK | "ask before you act" checks | 15 min |
| E | SHACL shapes | hard constraints that **report violations** | 15 min |
| F | SWRL rules | business rules that classify actions | 10 min |
| G | Putting it together | an ALLOW / BLOCK gate for agent proposals | 5 min |

✏️ cells marked **YOUR TURN** are for you. Replace the `____` blanks, run, and use the ✅/❌ checks.

## 0 · Setup

In [ ]:
# Run once. (Colab / local Jupyter both fine.)
# owlready2 is pinned to 0.47: newer releases bundle a Pellet build that needs Java 25.
%pip install -q rdflib owlrl pyshacl "owlready2==0.47" networkx matplotlib scipy

In [ ]:
import shutil, warnings
import networkx as nx
import matplotlib.pyplot as plt
from rdflib import Graph, Namespace, Literal, URIRef, RDF, RDFS, OWL, XSD
import owlrl
from pyshacl import validate
warnings.filterwarnings("ignore")

EX = Namespace("http://example.org/shop#")
PREFIXES = """
@prefix ex:   <http://example.org/shop#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix sh:   <http://www.w3.org/ns/shacl#> .
"""

# ── helpers ────────────────────────────────────────────────────────────────
def load(*turtle_chunks):
    """Parse one or more Turtle strings into a single Graph (prefixes added for you)."""
    g = Graph(); g.bind("ex", EX)
    for t in turtle_chunks:
        g.parse(data=PREFIXES + t, format="turtle")
    return g

def short(term):
    s = str(term)
    for ns, p in [(str(EX), ""), (str(RDF), "rdf:"), (str(RDFS), "rdfs:"), (str(OWL), "owl:")]:
        if s.startswith(ns): return p + s[len(ns):]
    return s

def interesting(g):
    """Only triples about our world: ex: subjects, ex: predicates or rdf:type to an ex: class."""
    for s, p, o in g:
        if not str(s).startswith(str(EX)): continue
        if p == RDF.type:
            if str(o).startswith(str(EX)): yield (s, p, o)
        elif str(p).startswith(str(EX)):
            yield (s, p, o)

def new_triples(before, after, show=True):
    """Triples in `after` that were not in `before` (i.e. what reasoning added)."""
    new = sorted(set(interesting(after)) - set(before), key=lambda t: (short(t[1]), short(t[0])))
    if show:
        print(f"✨ {len(new)} new triples inferred:")
        for s, p, o in new:
            print(f"   {short(s):10} {('a' if p == RDF.type else short(p)):12} {short(o)}")
    return new

def reason(g, level="rdfs"):
    """Return a NEW graph = g + everything RDFS or OWL-RL reasoning can infer."""
    g2 = Graph(); g2.bind("ex", EX)
    for t in g: g2.add(t)
    sem = owlrl.RDFS_Semantics if level == "rdfs" else owlrl.OWLRL_Semantics
    owlrl.DeductiveClosure(sem).expand(g2)
    return g2

def draw(g, base=None, title="", hide_types=False, size=(12, 7)):
    """Draw the graph. If `base` is given, triples NOT in base are drawn as red dashed (= inferred)."""
    G, labels, inferred = nx.DiGraph(), {}, {}
    classes = {o for _, p, o in g if p == RDF.type}
    for s, p, o in interesting(g):
        if hide_types and p == RDF.type: continue
        a, b = short(s), (f'"{o}"' if isinstance(o, Literal) else short(o))
        G.add_edge(a, b)
        labels.setdefault((a, b), []).append("a" if p == RDF.type else short(p))
        inferred[(a, b)] = inferred.get((a, b), False) or (base is not None and (s, p, o) not in base)
    colors = ["#f6c177" if n in {short(c) for c in classes} else
              "#d9d9d9" if n.startswith('"') else "#9ccfd8" for n in G.nodes]
    pos = {n: list(xy) for n, xy in nx.kamada_kawai_layout(G.to_undirected()).items()}
    nodes = list(pos)
    for _ in range(300):                      # nudge apart any nodes that overlap
        moved = False
        for i, a in enumerate(nodes):
            for b in nodes[i + 1:]:
                dx, dy = pos[b][0] - pos[a][0], pos[b][1] - pos[a][1]
                d = (dx * dx + dy * dy) ** 0.5 or 1e-3
                if d < 0.38:
                    f = (0.38 - d) / 2 / d; moved = True
                    pos[a][0] -= dx * f; pos[a][1] -= dy * f; pos[b][0] += dx * f; pos[b][1] += dy * f
        if not moved: break
    plt.figure(figsize=size)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1500, edgecolors="#555")
    nx.draw_networkx_labels(G, pos, font_size=9)
    for style, want in [("solid", False), ("dashed", True)]:
        for two_way in (False, True):          # curve two-way pairs apart so they don't overlap
            es = [e for e in G.edges if inferred[e] == want and G.has_edge(e[1], e[0]) == two_way]
            cs = "arc3,rad=0.3" if two_way else "arc3,rad=0.05"
            nx.draw_networkx_edges(G, pos, edgelist=es, style=style, arrowsize=18, node_size=1500,
                                   edge_color="#d7263d" if want else "#555", connectionstyle=cs)
            nx.draw_networkx_edge_labels(G, pos, {e: "\n".join(labels[e]) for e in es}, font_size=8,
                                         font_color="#d7263d" if want else "#333", connectionstyle=cs,
                                         bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    plt.title(title + ("   (red dashed = inferred)" if base is not None else "")); plt.axis("off"); plt.show()

def ask(g, q):
    try: return g.query(PREFIXES.replace("@prefix", "PREFIX").replace(" .", "") + q).askAnswer
    except Exception as e: print("⚠️  query error:", e)

def select(g, q, show=True):
    try: res = g.query(PREFIXES.replace("@prefix", "PREFIX").replace(" .", "") + q)
    except Exception as e: print("⚠️  query error:", e); return []
    rows = [tuple(short(x) if x is not None else "-" for x in r) for r in res]
    if show:
        print(" | ".join(str(v) for v in res.vars)); print("-" * 40)
        for r in rows: print(" | ".join(r))
    return rows

def check(label, ok):
    print(("✅ " if ok else "❌ ") + label)

print("Ready ✔   Java available for SWRL (part F):", bool(shutil.which("java")))

---
## A · Triples → Graph  *(10 min)*

Everything in RDF is a **triple**: `subject — predicate — object`.
A set of triples *is* a graph: subjects/objects are nodes, predicates are labelled arrows.

We write triples in **Turtle**. `;` means "same subject, next predicate". `a` is shorthand for `rdf:type`.
Here is what our support agent knows right now:

In [ ]:
KB_A = """
ex:anna     ex:placed      ex:order7 .
ex:order7   ex:contains    ex:laptop99 .

ex:refund42 ex:forOrder    ex:order7 ;
            ex:requestedBy ex:anna ;
            ex:approvedBy  ex:botX ;
            ex:amount      1200 .

ex:botX     a ex:AIAgent ;     ex:reportsTo ex:maria .
ex:maria    a ex:Manager ;     ex:reportsTo ex:gabor .
ex:gabor    a ex:Manager .
"""
g = load(KB_A)
print(len(g), "triples")
for s, p, o in g: print(f"  ({short(s)}, {short(p)}, {short(o)})")
draw(g, title="What botX knows")

### ✏️ YOUR TURN A — add a second customer
Write triples for this story, then run the cell to redraw:

> **Bence** placed **order8**, which contains **phone3**. He requested **refund43** for order8,
> amount **80**, and it was approved by **maria**.

In [ ]:
MY_A = """
ex:bence    ex:____  ex:____ .
ex:____     ex:____  ex:____ .

ex:refund43 ex:____  ex:____ ;
            ex:____  ex:____ ;
            ex:____  ex:____ ;
            ex:____  0 .      # ← property and amount
"""
try:
    g_mine = load(KB_A, MY_A)
    draw(g_mine, title="With Bence added")
    for t, lbl in [((EX.bence, EX.placed, EX.order8), "bence placed order8"),
                   ((EX.order8, EX.contains, EX.phone3), "order8 contains phone3"),
                   ((EX.refund43, EX.approvedBy, EX.maria), "refund43 approvedBy maria"),
                   ((EX.refund43, EX.amount, Literal(80)), "refund43 amount 80")]:
        check(lbl, t in g_mine)
except Exception as e:
    print("⚠️  Turtle didn't parse — check your blanks:", e)

#### 🔄 Sync point
So everyone continues from the same place, here is the **shared knowledge base** used from now on
(it includes the answer to A):

In [ ]:
KB = KB_A + """
ex:bence    ex:placed    ex:order8 .
ex:order8   ex:contains  ex:phone3 .
ex:refund43 ex:forOrder ex:order8 ; ex:requestedBy ex:bence ; ex:approvedBy ex:maria ; ex:amount 80 .
"""
print(len(load(KB)), "triples in the shared KB")

---
## B · RDFS: domain & range  *(15 min)*

RDFS lets us say what *kind* of thing sits at each end of a property:

* `ex:placed rdfs:domain ex:Customer` → **whoever** places something is a Customer
* `ex:placed rdfs:range  ex:Order`    → **whatever** is placed is an Order
* `ex:Manager rdfs:subClassOf ex:Employee` → every Manager is an Employee

A reasoner then **adds** the types nobody wrote down. Example first:

In [ ]:
SCHEMA_DEMO = """
ex:placed   rdfs:domain ex:Customer ;  rdfs:range ex:Order .

ex:Customer rdfs:subClassOf ex:Person .
ex:Employee rdfs:subClassOf ex:Person .
ex:Manager  rdfs:subClassOf ex:Employee .
ex:AIAgent  rdfs:subClassOf ex:Employee .
"""
g0  = load(KB)
g1  = reason(load(KB, SCHEMA_DEMO), "rdfs")
new_triples(g0, g1)
draw(g1, base=g0, title="RDFS inferences from `placed` + subclasses", size=(13, 8))

### ✏️ YOUR TURN B — fill in domain and range
Pick classes from: `Customer  Employee  Manager  Order  Product  Refund  Person`.
Think: *what kind of thing is on the left of this arrow? on the right?*

In [ ]:
MY_SCHEMA = """
ex:contains    rdfs:domain ex:____ ;  rdfs:range ex:____ .
ex:forOrder    rdfs:domain ex:____ ;  rdfs:range ex:____ .
ex:requestedBy rdfs:domain ex:____ ;  rdfs:range ex:____ .
ex:approvedBy  rdfs:domain ex:____ ;  rdfs:range ex:____ .
ex:reportsTo   rdfs:domain ex:____ ;  rdfs:range ex:____ .
"""
g_b = reason(load(KB, SCHEMA_DEMO, MY_SCHEMA), "rdfs")
new_triples(g0, g_b)
for (s, c) in [("laptop99", "Product"), ("refund42", "Refund"), ("order8", "Order"),
               ("bence", "Customer"), ("maria", "Person")]:
    check(f"{s} is a {c}", (EX[s], RDF.type, EX[c]) in g_b)

In [ ]:
draw(g_b, base=g0, title="All RDFS inferences", size=(15, 10))

### ⚠️ The gotcha: RDFS **infers**, it does not **reject**
Suppose the LLM agent hallucinates this action: *anna approves her own €5000 refund.*

In [ ]:
BAD = """
ex:refund44 ex:requestedBy ex:anna ; ex:approvedBy ex:anna ; ex:amount 5000 .
"""
g_bad = reason(load(KB, SCHEMA_DEMO, "ex:approvedBy rdfs:range ex:Employee .", BAD), "rdfs")
print("Is anna now an Employee?", (EX.anna, RDF.type, EX.Employee) in g_bad)
print("Any error raised?        No. 🙃")

RDFS works under the **Open-World Assumption**: anything not forbidden might be true, so it
cheerfully concludes that *anna must be an employee*. Useful for **filling gaps**, useless for **blocking**.
Blocking needs a closed-world check → that is SHACL (part E). Hold on.

---
## C · OWL: transitive & inverse properties  *(15 min)*

OWL adds richer property semantics. Two workhorses:

| Axiom | Meaning | Agent-guardrail use |
|---|---|---|
| `ex:p a owl:TransitiveProperty` | p(x,y) ∧ p(y,z) ⇒ p(x,z) | escalation / approval chains, part-of hierarchies |
| `ex:p owl:inverseOf ex:q` | p(x,y) ⇔ q(y,x) | navigate either direction without the agent storing both |

Demo — an inverse:

In [ ]:
OWL_DEMO = """
ex:placedBy owl:inverseOf ex:placed .
"""
g_c0 = reason(load(KB, OWL_DEMO), "owl")
new_triples(load(KB), g_c0)

### ✏️ YOUR TURN C
1. Make `ex:reportsTo` **transitive** (so escalation follows the whole chain).
2. Declare a new property `ex:manages` as the **inverse** of `ex:reportsTo`.

Before running: predict — *who will gabor manage?*

In [ ]:
MY_OWL = """
ex:reportsTo a owl:____ .
ex:manages   owl:____ ex:____ .
"""
g_c = reason(load(KB, OWL_DEMO, MY_OWL), "owl")
new_triples(load(KB), g_c)
check("botX reportsTo gabor (transitivity)", (EX.botX, EX.reportsTo, EX.gabor) in g_c)
check("maria manages botX (inverse)",        (EX.maria, EX.manages, EX.botX) in g_c)
check("gabor manages botX (both combined!)", (EX.gabor, EX.manages, EX.botX) in g_c)

In [ ]:
draw(g_c, base=load(KB), hide_types=True, title="OWL inferences (types hidden)", size=(13, 8))

---
## D · SPARQL: SELECT and ASK  *(15 min)*

SPARQL is SQL for graphs: you write a **triple pattern with variables** (`?x`) and the engine
finds every way to match it.

* `SELECT` → return the matches (a table)
* `ASK`    → just **true / false** — the natural shape for a guardrail: *"is this allowed?"*

From here on we query the **fully reasoned** knowledge base (RDFS + OWL from parts B & C).

In [ ]:
SCHEMA = SCHEMA_DEMO + """
ex:contains    rdfs:domain ex:Order ;     rdfs:range ex:Product .
ex:forOrder    rdfs:domain ex:Refund ;    rdfs:range ex:Order .
ex:requestedBy rdfs:domain ex:Refund ;    rdfs:range ex:Customer .
ex:approvedBy  rdfs:domain ex:Refund ;    rdfs:range ex:Employee .
ex:reportsTo   rdfs:domain ex:Employee ;  rdfs:range ex:Manager .
ex:placedBy    owl:inverseOf ex:placed .
ex:reportsTo   a owl:TransitiveProperty .
ex:manages     owl:inverseOf ex:reportsTo .
"""
raw = load(KB, SCHEMA)
kb  = reason(raw, "owl")
print(len(raw), "asserted triples →", len(kb), "after reasoning")

**Demo SELECT** — every refund, its amount, who approved it, and what kind of approver they are:

In [ ]:
select(kb, """
SELECT ?refund ?amount ?approver ?kind WHERE {
    ?refund   a ex:Refund ;
              ex:amount ?amount ;
              ex:approvedBy ?approver .
    ?approver a ?kind .
    FILTER (?kind IN (ex:AIAgent, ex:Manager))
}""")

**Demo ASK** — the same question on raw vs. reasoned data:

In [ ]:
q = "ASK { ex:botX ex:reportsTo ex:gabor }"
print("raw graph      :", ask(raw, q))
print("reasoned graph :", ask(kb,  q))
print("raw + property path reportsTo+ :", ask(raw, "ASK { ex:botX ex:reportsTo+ ex:gabor }"))

### ✏️ YOUR TURN D — write your own queries (on `kb`)
* **D1 (SELECT)** Who are *all* the people botX can escalate to? (expected: maria, gabor)
* **D2 (ASK — a guardrail!)** Has **any AI agent** approved a refund **over 1000**? (expected: `True` 🚨)
* **D3 (SELECT, stretch)** Which customer bought which product? Hint: property path `ex:placed/ex:contains`.

In [ ]:
d1 = select(kb, """
SELECT ?boss WHERE {
    ____
}""")
check("D1", sorted(d1) == [("gabor",), ("maria",)])

In [ ]:
d2 = ask(kb, """
ASK {
    ____
    FILTER (____)
}""")
print("Answer:", d2); check("D2", d2 is True)

In [ ]:
d3 = select(kb, """
SELECT ?customer ?product WHERE {
    ____
}""")
check("D3", sorted(d3) == [("anna", "laptop99"), ("bence", "phone3")])

---
## E · SHACL: shapes that report violations  *(15 min)*

RDFS/OWL **infer**. SHACL **validates**: it checks data against *shapes* under a
**closed-world** view — if a required thing is missing or wrong, you get a **violation report**.
This is the piece that turns an ontology into an actual guardrail.

The agent now proposes a new action (refund44 from part B). Let's validate it:

In [ ]:
PROPOSAL = """
ex:refund44 a ex:Refund ;
            ex:requestedBy ex:anna ;
            ex:approvedBy  ex:anna ;
            ex:amount      5000 .
"""
SHAPES_DEMO = """
ex:RefundShape a sh:NodeShape ;
    sh:targetClass ex:Refund ;
    sh:property [
        sh:path ex:amount ;
        sh:minCount 1 ;
        sh:datatype xsd:integer ;
        sh:maxInclusive 1000 ;
        sh:message "Refund amount must be ≤ €1000" ] ;
    sh:property [
        sh:path ex:forOrder ;
        sh:minCount 1 ;
        sh:message "Refund must reference an order" ] .
"""

def shacl(data_ttl_list, shapes_ttl, show=True):
    data   = load(*data_ttl_list)
    shapes = load(shapes_ttl)
    conforms, report, _ = validate(data, shacl_graph=shapes, inference="rdfs")
    rows = []
    for r in report.subjects(RDF.type, URIRef("http://www.w3.org/ns/shacl#ValidationResult")):
        SH = Namespace("http://www.w3.org/ns/shacl#")
        rows.append((short(report.value(r, SH.focusNode)), short(report.value(r, SH.resultPath)),
                     short(report.value(r, SH.value) or "-"), str(report.value(r, SH.resultMessage))))
    if show:
        print("CONFORMS ✅" if conforms else f"{len(rows)} VIOLATION(S) 🚫")
        for row in sorted(rows): print("   {:10} {:12} {:8} {}".format(*row))
    return conforms, sorted(rows)

shacl([KB, SCHEMA, PROPOSAL], SHAPES_DEMO);

### ✏️ YOUR TURN E — add two constraints to the shape
1. The approver must be a **Manager** (AI agents may not approve). Hint: `sh:class`
2. The approver must **not be the same person** who requested it. Hint: `sh:disjoint` compares two properties.

In [ ]:
MY_SHAPES = SHAPES_DEMO.rstrip().rstrip(".") + """ ;
    sh:property [
        sh:path ex:approvedBy ;
        sh:____ ex:____ ;
        sh:message "Approver must be a Manager" ] ;
    sh:property [
        sh:path ex:approvedBy ;
        sh:____ ex:____ ;
        sh:message "Nobody may approve their own refund" ] .
"""
ok, rows = shacl([KB, SCHEMA, PROPOSAL], MY_SHAPES)
check("botX (AI agent) caught approving refund42", ("refund42", "approvedBy", "botX", "Approver must be a Manager") in rows)
check("anna caught approving her own refund44", any(r[0] == "refund44" and "own" in r[3] for r in rows))

---
## F · SWRL: rules  *(10 min)*

Some business logic is naturally an **if-then rule**, not a type or a shape:

```
Refund(?r) ∧ amount(?r, ?a) ∧ greaterThan(?a, 1000)  →  HighValueRefund(?r)
```

**SWRL** (Semantic Web Rule Language) adds such rules to an OWL ontology. The reasoner (Pellet, via
`owlready2`, needs Java) fires them and **classifies** individuals — creating new categories your
guardrails can then act on.

In [ ]:
from owlready2 import (World, Thing, ObjectProperty, DataProperty, FunctionalProperty,
                       Imp, sync_reasoner_pellet)

def run_rules(rules, show=True):
    """Build a fresh ontology of our shop, add SWRL rules, run Pellet, return the ontology."""
    w = World()
    onto = w.get_ontology("http://example.org/shop#")
    with onto:
        class Person(Thing): pass
        class Customer(Person): pass
        class Employee(Person): pass
        class Manager(Employee): pass
        class AIAgent(Employee): pass
        class Refund(Thing): pass
        # categories the rules may fill in:
        class HighValueRefund(Refund): pass
        class SelfApproved(Refund): pass
        class NeedsHumanReview(Refund): pass
        class requestedBy(ObjectProperty): pass
        class approvedBy(ObjectProperty):  pass
        class reportsTo(ObjectProperty):   pass
        class escalateTo(ObjectProperty):  pass
        class amount(DataProperty, FunctionalProperty): range = [int]

        anna, bence = Customer("anna"), Customer("bence")
        maria, gabor = Manager("maria"), Manager("gabor")
        botX = AIAgent("botX"); botX.reportsTo = [maria]; maria.reportsTo = [gabor]
        Refund("refund42", amount=1200, requestedBy=[anna],  approvedBy=[botX])
        Refund("refund43", amount=80,   requestedBy=[bence], approvedBy=[maria])
        Refund("refund44", amount=5000, requestedBy=[anna],  approvedBy=[anna])

        for rule in rules:
            try: Imp().set_as_rule(rule)
            except Exception as e: print("⚠️  could not parse rule:", rule, "\n   ", e); return onto
    try:
        sync_reasoner_pellet(w, infer_property_values=True, infer_data_property_values=True, debug=0)
    except Exception as e:
        print("⚠️  Pellet failed (is Java installed? see note below):", str(e)[:300]); return onto
    if show:
        for cls in [onto.HighValueRefund, onto.SelfApproved, onto.NeedsHumanReview]:
            print(f"{cls.name:17}: {sorted(i.name for i in cls.instances())}")
        for r in onto.Refund.instances():
            if r.escalateTo: print(f"{r.name} escalateTo {[m.name for m in r.escalateTo]}")
    return onto

DEMO_RULES = [
    "Refund(?r), amount(?r, ?a), greaterThan(?a, 1000) -> HighValueRefund(?r)",
    "Refund(?r), approvedBy(?r, ?e), reportsTo(?e, ?m) -> escalateTo(?r, ?m)",
]
onto = run_rules(DEMO_RULES)

> **No Java?** On Colab run `!apt-get -qq install -y openjdk-17-jre-headless` and restart.
> The next cell shows the *same* first rule as a SPARQL `CONSTRUCT` — a rule is really just
> "match a pattern → add triples", which works anywhere.

In [ ]:
rule_as_sparql = PREFIXES.replace("@prefix", "PREFIX").replace(" .", "") + """
CONSTRUCT { ?r a ex:HighValueRefund }
WHERE     { ?r a ex:Refund ; ex:amount ?a . FILTER (?a > 1000) }
"""
for s, p, o in kb.query(rule_as_sparql): print(short(s), "a", short(o))

### ✏️ YOUR TURN F — write two rules
* **F1** A refund is `SelfApproved` if the same person requested *and* approved it.
  (Hint: reuse the same variable `?p` in two atoms.)
* **F2** A refund `NeedsHumanReview` if it is a `HighValueRefund` **and** it was approved by an `AIAgent`.
  (This rule *chains* on the demo rule's output.)

In [ ]:
MY_RULES = [
    "____ -> SelfApproved(?r)",
    "____ -> NeedsHumanReview(?r)",
]
onto = run_rules(DEMO_RULES + MY_RULES)
if onto.SelfApproved:
    check("F1 SelfApproved = refund44",     sorted(i.name for i in onto.SelfApproved.instances()) == ["refund44"])
    check("F2 NeedsHumanReview = refund42", sorted(i.name for i in onto.NeedsHumanReview.instances()) == ["refund42"])

---
## G · Putting it together: a guardrail gate for agent actions  *(5 min)*

The LLM agent emits its intended action **as triples**. Before execution we:
1. merge it with the knowledge base + ontology,
2. **reason** (RDFS + OWL) to fill in what's implied,
3. **validate** with SHACL → ALLOW, or BLOCK with reasons the agent can read and fix.

In [ ]:
GUARDRAIL_SHAPES = """
ex:RefundShape a sh:NodeShape ; sh:targetClass ex:Refund ;
    sh:property [ sh:path ex:amount ; sh:minCount 1 ; sh:maxInclusive 1000 ;
                  sh:message "Refund amount must be ≤ €1000" ] ;
    sh:property [ sh:path ex:forOrder ; sh:minCount 1 ; sh:message "Refund must reference an order" ] ;
    sh:property [ sh:path ex:approvedBy ; sh:class ex:Manager ; sh:message "Approver must be a Manager" ] ;
    sh:property [ sh:path ex:approvedBy ; sh:disjoint ex:requestedBy ;
                  sh:message "Nobody may approve their own refund" ] .
"""
def gate(proposal_ttl):
    proposed = set(load(proposal_ttl).subjects())               # the new things the agent wants to create
    data = reason(load(KB, SCHEMA, proposal_ttl), "owl")         # context + ontology + proposal, reasoned
    _, report, _ = validate(data, shacl_graph=load(GUARDRAIL_SHAPES))
    SH = Namespace("http://www.w3.org/ns/shacl#")
    msgs = sorted({str(report.value(r, SH.resultMessage))        # only judge the proposal, not old history
                   for r in report.subjects(RDF.type, SH.ValidationResult)
                   if report.value(r, SH.focusNode) in proposed})
    return "🚫 BLOCK — " + "; ".join(msgs) if msgs else "✅ ALLOW"

# Three actions an LLM agent might propose:
proposals = {
    "maria refunds €50 for order8":
        "ex:r100 a ex:Refund ; ex:forOrder ex:order8 ; ex:requestedBy ex:bence ; ex:approvedBy ex:maria ; ex:amount 50 .",
    "botX refunds €1500 for order7":
        "ex:r101 a ex:Refund ; ex:forOrder ex:order7 ; ex:requestedBy ex:anna ; ex:approvedBy ex:botX ; ex:amount 1500 .",
    "gabor refunds himself":
        "ex:r102 a ex:Refund ; ex:forOrder ex:order7 ; ex:requestedBy ex:gabor ; ex:approvedBy ex:gabor ; ex:amount 20 .",
}
for name, ttl in proposals.items():
    print(f"{name:32} → {gate(ttl)}")

### 🎯 Take-aways
| Layer | Question it answers for the agent |
|---|---|
| **Triples** | What exists, stated explicitly? |
| **RDFS** | What *kinds* of things are these? (fills gaps, never blocks) |
| **OWL** | What else follows — chains, inverses? |
| **SPARQL ASK** | Is this specific condition true *right now*? |
| **SHACL** | Does this action break policy? → ALLOW / BLOCK + reasons |
| **SWRL** | Which business category does this fall into? |

**The LLM proposes, the ontology disposes.** 🧭

*Next steps:* expose `gate()` as a tool/MCP server the agent must call before any write action,
and feed the SHACL messages back into the agent's context so it can repair its own plan.